In [1]:
from pathlib import Path
from math import gcd
from time import perf_counter
import importlib
import inspect
import json
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly

In [2]:
EVALUATION_SPLIT = "test"       
MAX_SAMPLES = 12
NUM_BACKGROUND_PILOT = 6
NUM_FRAMES = 4
DOWNLOAD_MISSING_MP4 = True
USE_CUDA = True
RANDOM_SEED = 42
MAMMALPS_ZIP_URL = "https://zenodo.org/records/15588220/files/mammalps_v1.zip"

# auto: trained weights if checkpoint found, else untrained model for compute profiling
# trained: fail if checkpoint missing; profile_only: always use untrained model
VIDEO_MODE = "auto"  # "auto", "trained", "profile_only"
VIDEO_ADAPTIVE_THRESHOLD = None
VIDEO_CHECKPOINT_OVERRIDE = None
VIDEO_CODE_DIR_OVERRIDE = None


def find_project_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "data" / "metadata" / "benchmark_2_audio_model_manifest.csv").is_file():
            return base.resolve()
    raise FileNotFoundError("Run from project root or Audio/: missing data/metadata/benchmark_2_audio_model_manifest.csv")

ROOT = find_project_root()
AUDIO_DIR = ROOT / "Audio"
if not AUDIO_DIR.is_dir():
    AUDIO_DIR = ROOT / "audio"
MODELS_DIR = AUDIO_DIR / "models"
RESULTS_DIR = AUDIO_DIR / "results"
MANIFEST_PATH = ROOT / "data" / "metadata" / "benchmark_2_audio_model_manifest.csv"
VIDEO_CACHE_DIR = ROOT / "data" / "video_cache" / "raw"
OUTPUT_DIR = AUDIO_DIR / "results" / "stage_a"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device("cuda" if USE_CUDA and torch.cuda.is_available() else "cpu")
print("Root:", ROOT)
print("Device:", DEVICE)
print("Video cache:", VIDEO_CACHE_DIR)

Root: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project
Device: cuda
Video cache: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\data\video_cache\raw


In [4]:
manifest = pd.read_csv(MANIFEST_PATH)
required = {
    "sample_id", "event_id", "file_id", "remote_video_path", "start_2s", "end_2s",
    "duration", "binary_label", "dataset_split",
}
assert required.issubset(manifest.columns), f"Manifest missing: {sorted(required - set(manifest.columns))}"
assert manifest["sample_id"].is_unique
assert np.allclose(manifest.end_2s - manifest.start_2s, 2.0, atol=0.001)
assert not manifest[list(required)].isna().any().any(), "Missing required manifest fields"
assert manifest.groupby("event_id").dataset_split.nunique().max() == 1

pool = manifest[manifest.dataset_split == EVALUATION_SPLIT].copy()
assert len(pool) > 0, f"No rows for split={EVALUATION_SPLIT!r}"

# Detect direct recording overlap with teammate's Benchmark I train metadata.
b1_train_path = ROOT / "data" / "metadata" / "train.csv"
if b1_train_path.exists():
    b1train = pd.read_csv(b1_train_path)
    assert "video_path" in b1train.columns
    b1_train_raw_ids = set(
        b1train["video_path"].astype(str).str.extract(r"(S\d+_C\d+_E\d+_V\d+)")[0].dropna()
    )
    overlap = pool.file_id.isin(b1_train_raw_ids)
    print("Test/validation audio samples from a video-training recording:", int(overlap.sum()))
    # Prevent using contaminated records in this demo; don't relabel or re-split models.
    pool = pool.loc[~overlap].copy()
else:
    warnings.warn("Benchmark I data/metadata/train.csv missing: video-training overlap NOT checked")

if MAX_SAMPLES is not None:
    assert MAX_SAMPLES > 0
    bg = pool[pool.binary_label == 0].sample(n=min(NUM_BACKGROUND_PILOT, (pool.binary_label == 0).sum(), MAX_SAMPLES), random_state=RANDOM_SEED)
    animals = pool[pool.binary_label == 1].sample(n=min(MAX_SAMPLES - len(bg), (pool.binary_label == 1).sum()), random_state=RANDOM_SEED)
    selected = pd.concat([bg, animals], ignore_index=True)
else:
    selected = pool.reset_index(drop=True)

assert len(selected), "No samples selected"
print("Audio manifest total:", len(manifest))
print("Evaluation pool (after overlap filter):", len(pool))
print("Selected pilot size:", len(selected))
print(selected.binary_label.value_counts().rename({0:"background", 1:"animal"}).to_string())
display(selected[["sample_id", "file_id", "start_2s", "end_2s", "binary_label", "remote_video_path"]].head(12))

Test/validation audio samples from a video-training recording: 0
Audio manifest total: 908
Evaluation pool (after overlap filter): 207
Selected pilot size: 12
binary_label
background    6
animal        6


,sample_id,file_id,start_2s,end_2s,binary_label,remote_video_path
0,E607_1,S3_C3_E607_V0568,44.375000,46.375000,0,raw_videos/S3/C3/S3_C3_E607_V0568.mp4
1,E297_0,S2_C2_E297_V0048,29.125000,31.125000,0,raw_videos/S2/C2/S2_C2_E297_V0048.mp4
2,E549_1,S3_C3_E549_V0444,29.250000,31.250000,0,raw_videos/S3/C3/S3_C3_E549_V0444.mp4
3,E507_1,S3_C3_E507_V0292,29.250000,31.250000,0,raw_videos/S3/C3/S3_C3_E507_V0292.mp4
4,E504_0,S3_C3_E504_V0288,14.125000,16.125000,0,raw_videos/S3/C3/S3_C3_E504_V0288.mp4
5,E604_1,S3_C3_E604_V0564,29.250000,31.250000,0,raw_videos/S3/C3/S3_C3_E604_V0564.mp4
6,E153_0,S1_C2_E153_V0383,1.983333,3.983333,1,raw_videos/S1/C2/S1_C2_E153_V0383.mp4
7,E301_1,S2_C2_E301_V0052,35.983333,37.983333,1,raw_videos/S2/C2/S2_C2_E301_V0052.mp4
8,E304_0,S2_C1_E304_V0050,21.300000,23.300000,1,raw_videos/S2/C1/S2_C1_E304_V0050.mp4
9,E61_0,S1_C3_E61_V0029,8.466667,10.466667,1,raw_videos/S1/C3/S1_C3_E61_V0029.mp4


In [5]:
paths = {
    "PANNs pretrained backbone": MODELS_DIR / "pretrained" / "Cnn14_mAP=0.431.pth",
    "Final audio head": MODELS_DIR / "panns_cnn14_binary_head_2p0s.pt",
    "Audio exit 1": MODELS_DIR / "panns_exit1_block2_2p0s.pt",
    "Audio exit 2": MODELS_DIR / "panns_exit2_block4_2p0s.pt",
    "Frozen policy": RESULTS_DIR / "panns_early_exit_policy_results_2p0s.csv",
    "Frozen thresholds": RESULTS_DIR / "panns_early_exit_standalone_2p0s.csv",
    "Audio profiled FLOPs": RESULTS_DIR / "panns_audio_all_paths_flops_2p0s.csv",
}
video_code_choices = [ROOT, ROOT / "Video", ROOT / "video"]
VIDEO_CODE_DIR = (Path(VIDEO_CODE_DIR_OVERRIDE) if VIDEO_CODE_DIR_OVERRIDE else
                  next((p for p in video_code_choices if (p / "video.py").is_file()), None))
video_checkpoint_choices = [
    ROOT / "checkpoints" / "activity_baseline.pt",
    ROOT / "Video" / "checkpoints" / "activity_baseline.pt",
    ROOT / "video" / "checkpoints" / "activity_baseline.pt",
    ROOT / "checkpoints" / "baseline.pt",
    ROOT / "Video" / "checkpoints" / "baseline.pt",
]
VIDEO_CHECKPOINT = (Path(VIDEO_CHECKPOINT_OVERRIDE) if VIDEO_CHECKPOINT_OVERRIDE else
                    next((p for p in video_checkpoint_choices if p.is_file()), video_checkpoint_choices[0]))

assert VIDEO_MODE in {"auto", "trained", "profile_only"}
VIDEO_WEIGHTS_AVAILABLE = VIDEO_CHECKPOINT.is_file()
VIDEO_INFERENCE_MODE = ("trained" if VIDEO_WEIGHTS_AVAILABLE else "profile_only") if VIDEO_MODE == "auto" else VIDEO_MODE

rows = [{"item": k, "found": p.is_file(), "path": str(p)} for k, p in paths.items()]
rows.append({"item": "video.py", "found": VIDEO_CODE_DIR is not None,
             "path": str(VIDEO_CODE_DIR / "video.py") if VIDEO_CODE_DIR else "NOT FOUND"})
rows.append({"item": "Video activity checkpoint (optional in auto/profile_only)",
             "found": VIDEO_WEIGHTS_AVAILABLE, "path": str(VIDEO_CHECKPOINT)})
preflight = pd.DataFrame(rows)
display(preflight)
required_missing = preflight.iloc[:-1].loc[~preflight.iloc[:-1].found, ["item", "path"]]
if len(required_missing):
    raise FileNotFoundError("Missing required pipeline inputs:\n" + required_missing.to_string(index=False))
if VIDEO_INFERENCE_MODE == "trained" and not VIDEO_WEIGHTS_AVAILABLE:
    raise FileNotFoundError(f"Trained video inference requires checkpoint: {VIDEO_CHECKPOINT}")
if VIDEO_ADAPTIVE_THRESHOLD is not None and VIDEO_INFERENCE_MODE != "trained":
    raise ValueError("Adaptive video routing requires a trained activity checkpoint")
print("VIDEO MODE:", VIDEO_INFERENCE_MODE.upper())
if VIDEO_INFERENCE_MODE == "profile_only":
    print("UNTRAINED VIDEO MODEL: valid for architecture-cost and wiring ONLY; no activity-quality conclusions")


,item,found,path
0,PANNs pretrained backbone,True,C:\Users\benja\Documents\CS_Master\Big Data at...
1,Final audio head,True,C:\Users\benja\Documents\CS_Master\Big Data at...
2,Audio exit 1,True,C:\Users\benja\Documents\CS_Master\Big Data at...
3,Audio exit 2,True,C:\Users\benja\Documents\CS_Master\Big Data at...
4,Frozen policy,True,C:\Users\benja\Documents\CS_Master\Big Data at...
5,Frozen thresholds,True,C:\Users\benja\Documents\CS_Master\Big Data at...
6,Audio profiled FLOPs,True,C:\Users\benja\Documents\CS_Master\Big Data at...
7,video.py,True,C:\Users\benja\Documents\CS_Master\Big Data at...
8,Video activity checkpoint (optional in auto/pr...,False,C:\Users\benja\Documents\CS_Master\Big Data at...


VIDEO MODE: PROFILE_ONLY
UNTRAINED VIDEO MODEL: valid for architecture-cost and wiring ONLY; no activity-quality conclusions


In [6]:
from urllib.request import urlretrieve
PANNS_LABELS = Path.home() / "panns_data" / "class_labels_indices.csv"
if not PANNS_LABELS.exists():
    PANNS_LABELS.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve("https://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv", PANNS_LABELS)

from panns_inference import AudioTagging

POLICY = pd.read_csv(paths["Frozen policy"])
row_policy = POLICY[(POLICY.operating_point == "balanced") & (POLICY.split == "validation")]
assert len(row_policy) == 1, "Expected exactly one balanced validation policy"
MARGIN1 = float(row_policy.iloc[0].margin1)
MARGIN2 = float(row_policy.iloc[0].margin2)

standalone = pd.read_csv(paths["Frozen thresholds"])
val_thresholds = standalone[standalone.split == "validation"]
def threshold_for(label):
    subset = val_thresholds[val_thresholds["exit"].str.lower().str.contains(label)]
    assert len(subset) == 1, f"Cannot find exactly one validation threshold for {label!r}"
    return float(subset.iloc[0].threshold)
T1, T2, T_FINAL = threshold_for("block 2"), threshold_for("block 4"), threshold_for("final")
print("Balanced policy margins:", MARGIN1, MARGIN2)
print("Frozen thresholds:", T1, T2, T_FINAL)

panns = AudioTagging(checkpoint_path=str(paths["PANNs pretrained backbone"]), device=DEVICE.type)
BACKBONE = panns.model.module if isinstance(panns.model, nn.DataParallel) else panns.model
BACKBONE.eval().requires_grad_(False)

class BinaryHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.30):
        super().__init__()
        self.network = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim, hidden_dim),
                                     nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))
    def forward(self, x):
        return self.network(x).squeeze(-1)

def load_head(path, state_field):
    ck = torch.load(path, map_location="cpu", weights_only=True)
    head = BinaryHead(ck["input_dim"], ck["hidden_dim"], ck["dropout"]).to(DEVICE).eval()
    head.load_state_dict(ck[state_field], strict=True)
    head.requires_grad_(False)
    return head

EXIT1_HEAD = load_head(paths["Audio exit 1"], "state_dict")
EXIT2_HEAD = load_head(paths["Audio exit 2"], "state_dict")
FINAL_HEAD = load_head(paths["Final audio head"], "head_state_dict")

def pooled(x):
    x = x.mean(dim=3)
    return x.max(dim=2).values + x.mean(dim=2)

def logit_margin(p, threshold):
    eps = 1e-6
    p, threshold = np.clip([p, threshold], eps, 1-eps)
    return float(np.log(p / (1-p)) - np.log(threshold / (1-threshold)))

@torch.inference_mode()
def balanced_audio_predict(waveform_32k):
    wav = torch.from_numpy(waveform_32k[None,:]).float().to(DEVICE)
    m = BACKBONE
    x = m.logmel_extractor(m.spectrogram_extractor(wav))
    x = m.bn0(x.transpose(1,3)).transpose(1,3)
    x = m.conv_block1(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block2(x, pool_size=(2,2), pool_type="avg")
    p1 = float(torch.sigmoid(EXIT1_HEAD(pooled(x))).item())
    d1 = logit_margin(p1, T1)
    if abs(d1) >= MARGIN1:
        return {"audio_prediction": int(d1 >= MARGIN1), "audio_probability": p1, "exit_stage": 1}
    x = m.conv_block3(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block4(x, pool_size=(2,2), pool_type="avg")
    p2 = float(torch.sigmoid(EXIT2_HEAD(pooled(x))).item())
    d2 = logit_margin(p2, T2)
    if abs(d2) >= MARGIN2:
        return {"audio_prediction": int(d2 >= MARGIN2), "audio_probability": p2, "exit_stage": 2}
    x = m.conv_block5(x, pool_size=(2,2), pool_type="avg")
    x = m.conv_block6(x, pool_size=(1,1), pool_type="avg")
    x = x.mean(dim=3)
    emb = F.relu(m.fc1(x.max(dim=2).values + x.mean(dim=2)))
    prob = float(torch.sigmoid(FINAL_HEAD(emb)).item())
    return {"audio_prediction": int(prob >= T_FINAL), "audio_probability": prob, "exit_stage": 3}

Balanced policy margins: 2.0 0.25
Frozen thresholds: 0.2324143648147583 0.2000872790813446 0.1661750376224517
Checkpoint path: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\models\pretrained\Cnn14_mAP=0.431.pth
GPU number: 1


In [7]:
AUDIO_SAMPLE_RATE = 32000
AUDIO_SECONDS = 2.0

def get_audio_path(row):
    return ROOT / "data" / "audio_cache" / "full" / f"{row['file_id']}.flac"

def read_audio_window(row):
    audio_path = get_audio_path(row)
    if not audio_path.is_file():
        raise FileNotFoundError(f"Cached soundtrack missing: {audio_path}")
    start, end = float(row["start_2s"]), float(row["end_2s"])
    assert abs((end - start) - AUDIO_SECONDS) <= 0.001
    with sf.SoundFile(str(audio_path)) as sfh:
        source_sr = int(sfh.samplerate)
        sfh.seek(round(start * source_sr))
        x = sfh.read(frames=round(AUDIO_SECONDS * source_sr),
                     dtype="float32", always_2d=False)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if source_sr != AUDIO_SAMPLE_RATE:
        d = gcd(source_sr, AUDIO_SAMPLE_RATE)
        x = resample_poly(x, up=AUDIO_SAMPLE_RATE // d, down=source_sr // d)
    expected = round(AUDIO_SECONDS * AUDIO_SAMPLE_RATE)
    if len(x) < expected:
        x = np.pad(x, (0, expected-len(x)))
    x = x[:expected]
    return np.clip(x, -1, 1).astype(np.float32)

print("Cached soundtracks present in selection:", sum(get_audio_path(r).is_file() for _,r in selected.iterrows()), "/", len(selected))

Cached soundtracks present in selection: 12 / 12


In [9]:
sys.path.insert(0, str(VIDEO_CODE_DIR.resolve()))
video_module = importlib.import_module("video")
from torchvision.models import MobileNet_V3_Large_Weights

if VIDEO_INFERENCE_MODE == "trained":
    video_ck = torch.load(str(VIDEO_CHECKPOINT), map_location="cpu", weights_only=True)
    if "model" in video_ck:
        raise ValueError("Full pickled model checkpoint not supported here; use the baseline state_dict checkpoint")
    if video_ck.get("target", "activity") != "activity":
        raise ValueError("Video checkpoint must be for activity classification")
    class_to_idx = video_ck.get("class_to_idx")
    if not class_to_idx:
        raise ValueError("Video checkpoint must store class_to_idx")
    idx_to_class = {int(v): k for k, v in class_to_idx.items()}
    assert sorted(idx_to_class) == list(range(len(idx_to_class)))
    training_frames = video_ck.get("num_frames", video_ck.get("config", {}).get("num_frames"))
    if training_frames is not None and int(training_frames) != NUM_FRAMES:
        warnings.warn(f"Video weights trained with {training_frames} frames, running {NUM_FRAMES}")
else:
    # Only head size is taken from training labels. NO pretrained/trained activity weights.
    b1_train_path = ROOT / "data" / "metadata" / "train.csv"
    if not b1_train_path.exists():
        raise FileNotFoundError("Profiling-only mode needs data/metadata/train.csv to obtain activity head shape")
    names = sorted(pd.read_csv(b1_train_path)["activity"].dropna().unique().tolist())
    if not names:
        raise ValueError("No activity labels in train.csv")
    class_to_idx = {name: j for j, name in enumerate(names)}
    idx_to_class = {j: name for name, j in class_to_idx.items()}

VIDEO_MODEL = video_module.Classifier(DEVICE, (1, NUM_FRAMES, 3, 224, 224), None,
                                     num_classes=len(idx_to_class))
if VIDEO_INFERENCE_MODE == "trained":
    VIDEO_MODEL.load_state_dict(video_ck.get("model_state_dict", video_ck), strict=True)
VIDEO_MODEL.to(DEVICE).eval().requires_grad_(False)
VIDEO_TRANSFORM = MobileNet_V3_Large_Weights.DEFAULT.transforms()
ADAPTIVE_SUPPORTED = {"adaptive_threshold", "return_frames_used"}.issubset(
    inspect.signature(VIDEO_MODEL.forward).parameters)
if VIDEO_ADAPTIVE_THRESHOLD is not None and not ADAPTIVE_SUPPORTED:
    raise ValueError("Current Video/video.py does not support adaptive_threshold; update it first")
print("Video mode:", VIDEO_INFERENCE_MODE)
print("Activity output dimensions:", len(idx_to_class))
print("Adaptive video code available:", ADAPTIVE_SUPPORTED)
if VIDEO_INFERENCE_MODE == "trained":
    print("Checkpoint loaded:", VIDEO_CHECKPOINT)
else:
    print("Warning: activity predictions disabled; classifier weights are UNTRAINED")


C:\Users\benja\anaconda3\envs\BDE\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Video mode: profile_only
Activity output dimensions: 11
Adaptive video code available: True


In [10]:
import av
from contextlib import contextmanager

@contextmanager
def remote_video_cache():
    opened = None
    try:
        def retrieve(row):
            nonlocal opened
            remote = str(row["remote_video_path"]).replace("\\", "/")
            path = VIDEO_CACHE_DIR / f"{row['file_id']}.mp4"
            if path.is_file() and path.stat().st_size > 0:
                return path
            if not DOWNLOAD_MISSING_MP4:
                raise FileNotFoundError(f"Raw video not cached: {path}")
            if opened is None:
                from remotezip import RemoteZip
                opened = RemoteZip(MAMMALPS_ZIP_URL)
            partial = path.with_suffix(".download")
            try:
                with opened.open(remote) as src, partial.open("wb") as dst:
                    shutil.copyfileobj(src, dst, length=1024*1024)
                if partial.stat().st_size == 0:
                    raise ValueError(f"Downloaded an empty video: {remote}")
                partial.replace(path)
            except Exception:
                partial.unlink(missing_ok=True)
                raise
            return path
        yield retrieve
    finally:
        if opened is not None:
            opened.close()

def synchronized_frames(video_path, start_s, end_s, num_frames=NUM_FRAMES):
    # Nested ordering must match the video model's adaptive progression.
    if VIDEO_ADAPTIVE_THRESHOLD is not None:
        positions = video_module.nested_frame_positions(num_frames)
        # Keep requested times inside the source interval (rather than full raw MP4).
        target_times = float(start_s) + (float(end_s) - float(start_s)) * np.asarray(positions)
    else:
        target_times = np.linspace(float(start_s), float(end_s), num_frames+2)[1:-1]
    best_delta = [float("inf")] * num_frames
    best_frame = [None] * num_frames
    with av.open(str(video_path)) as container:
        stream = container.streams.video[0]
        for frame in container.decode(video=0):
            sec = frame.time
            if sec is None and frame.pts is not None:
                sec = float(frame.pts * stream.time_base)
            if sec is None:
                continue
            for j, tgt in enumerate(target_times):
                delta = abs(float(sec) - float(tgt))
                if delta < best_delta[j]:
                    best_delta[j] = delta
                    best_frame[j] = frame.to_ndarray(format="rgb24")
            if sec > float(end_s) + 0.5 and all(f is not None for f in best_frame):
                break
    if any(f is None for f in best_frame):
        raise RuntimeError(f"Video has no timestamped frames: {video_path}")
    if max(best_delta) > 0.5:
        raise RuntimeError(f"Frames too far from requested times: {max(best_delta):.3f} s")
    return torch.stack([VIDEO_TRANSFORM(torch.from_numpy(rgb).permute(2, 0, 1))
                        for rgb in best_frame]).unsqueeze(0)

@torch.inference_mode()
def execute_video_model(frames):
    inp = frames.to(DEVICE)
    if ADAPTIVE_SUPPORTED:
        logits, counts = VIDEO_MODEL(
            inp, adaptive_threshold=VIDEO_ADAPTIVE_THRESHOLD, return_frames_used=True)
        frames_used = int(counts[0].item())
    else:
        logits = VIDEO_MODEL(inp)
        frames_used = int(inp.shape[1])
    if VIDEO_INFERENCE_MODE == "profile_only":
        # Explicitly suppress untrained activity labels/confidences.
        return {"activity_prediction": None, "activity_confidence": np.nan,
                "video_frames_used": frames_used}
    probs = logits.softmax(dim=-1)[0].detach().cpu().numpy()
    index = int(probs.argmax())
    return {"activity_prediction": idx_to_class[index],
            "activity_confidence": float(probs[index]),
            "video_frames_used": frames_used}


In [11]:
audio_flops = pd.read_csv(paths["Audio profiled FLOPs"])
flops_by_path = dict(zip(audio_flops.path, audio_flops.gflops_est))
needed = ("adaptive_exit1", "adaptive_exit2", "adaptive_final")
assert all(k in flops_by_path for k in needed)
AUDIO_GFLOPS = {1: float(flops_by_path["adaptive_exit1"]),
                2: float(flops_by_path["adaptive_exit2"]),
                3: float(flops_by_path["adaptive_final"])}

from thop import profile
with torch.inference_mode():
    one_frame = torch.zeros((1, 3, 224, 224), device=DEVICE)
    video_macs, _ = profile(VIDEO_MODEL.model, inputs=(one_frame,), verbose=False)
VIDEO_GFLOPS_PER_FRAME = 2.0 * float(video_macs) / 1e9
VIDEO_GFLOPS = VIDEO_GFLOPS_PER_FRAME * NUM_FRAMES
print("Audio GFLOPs by actual exit:", AUDIO_GFLOPS)
print(f"Video single-frame GFLOPs (estimated): {VIDEO_GFLOPS_PER_FRAME:.4f}")
print(f"Video {NUM_FRAMES}-frame static GFLOPs (estimated): {VIDEO_GFLOPS:.4f}")
assert VIDEO_GFLOPS > 0


Audio GFLOPs by actual exit: {1: 2.821031552, 2: 5.657237376, 3: 8.385780096}
Video single-frame GFLOPs (estimated): 0.4671
Video 4-frame static GFLOPs (estimated): 1.8686


In [12]:
def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# Warm up once so first-time CUDA initialization does not dominate the first sample.
if DEVICE.type == "cuda":
    with torch.inference_mode():
        _ = VIDEO_MODEL.model(torch.zeros((1, 3, 224, 224), device=DEVICE))
    sync_device()

records = []
with remote_video_cache() as ensure_video:
    for k, (_, row) in enumerate(selected.iterrows(), start=1):
        out = {"sample_id": row["sample_id"], "event_id": row["event_id"],
               "file_id": row["file_id"], "split": row["dataset_split"],
               "binary_label": int(row["binary_label"]),
               "start_2s": float(row["start_2s"]), "end_2s": float(row["end_2s"]),
               "video_inference_mode": VIDEO_INFERENCE_MODE,
               "video_status": "skipped", "activity_prediction": None,
               "activity_confidence": np.nan, "video_frames_used": 0,
               "video_download_ms": np.nan, "video_decode_ms": 0.0, "video_model_ms": 0.0}
        try:
            t0 = perf_counter()
            waveform = read_audio_window(row)
            out["audio_preprocess_ms"] = (perf_counter() - t0)*1000
            sync_device()
            t0 = perf_counter()
            decision = balanced_audio_predict(waveform)
            sync_device()
            out["audio_model_ms"] = (perf_counter() - t0)*1000
            out.update(decision)
            out["audio_gflops_est"] = AUDIO_GFLOPS[decision["exit_stage"]]
            out["video_activated"] = bool(decision["audio_prediction"])
            out["video_gflops_est"] = 0.0
            if out["video_activated"]:
                t0 = perf_counter()
                video_path = ensure_video(row)
                out["video_download_ms"] = (perf_counter() - t0)*1000
                t0 = perf_counter()
                frames = synchronized_frames(video_path, row["start_2s"], row["end_2s"])
                out["video_decode_ms"] = (perf_counter() - t0)*1000
                sync_device()
                t0 = perf_counter()
                out.update(execute_video_model(frames))
                sync_device()
                out["video_model_ms"] = (perf_counter() - t0)*1000
                out["video_gflops_est"] = VIDEO_GFLOPS_PER_FRAME * out["video_frames_used"]
                out["video_status"] = "executed"
            out["run_status"] = "ok"
        except Exception as exc:
            out["run_status"] = "error"
            out["error"] = f"{type(exc).__name__}: {exc}"
        out["total_model_gflops_est"] = out.get("audio_gflops_est", np.nan) + out.get("video_gflops_est", np.nan)
        out["local_total_latency_ms"] = (
            out.get("audio_preprocess_ms", 0) + out.get("audio_model_ms", 0) +
            out.get("video_decode_ms", 0) + out.get("video_model_ms", 0)
        )
        records.append(out)
        print(f"{k}/{len(selected)} {row['sample_id']} | "
              f"audio={out.get('audio_prediction','ERROR')} exit={out.get('exit_stage','?')} "
              f"vision={out['video_status']} frames={out['video_frames_used']} "
              f"{out.get('activity_prediction') or ''} {out.get('error','')}")

results = pd.DataFrame(records)
OUTPUT_PATH = OUTPUT_DIR / f"stage_a_predictions_{EVALUATION_SPLIT}_{len(selected)}_{VIDEO_INFERENCE_MODE}.csv"
results.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
display(results.head())


1/12 E607_1 | audio=0 exit=2 vision=skipped frames=0  
2/12 E297_0 | audio=1 exit=2 vision=executed frames=4  
3/12 E549_1 | audio=0 exit=2 vision=skipped frames=0  
4/12 E507_1 | audio=0 exit=2 vision=skipped frames=0  
5/12 E504_0 | audio=0 exit=2 vision=skipped frames=0  
6/12 E604_1 | audio=0 exit=2 vision=skipped frames=0  
7/12 E153_0 | audio=1 exit=1 vision=executed frames=4  
8/12 E301_1 | audio=1 exit=1 vision=executed frames=4  
9/12 E304_0 | audio=1 exit=2 vision=executed frames=4  
10/12 E61_0 | audio=1 exit=1 vision=executed frames=4  
11/12 E131_1 | audio=1 exit=1 vision=executed frames=4  
12/12 E293_1 | audio=1 exit=1 vision=executed frames=4  
Saved: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\results\stage_a\stage_a_predictions_test_12_profile_only.csv


,sample_id,event_id,file_id,split,binary_label,start_2s,end_2s,video_inference_mode,video_status,activity_prediction,...,audio_model_ms,audio_prediction,audio_probability,exit_stage,audio_gflops_est,video_activated,video_gflops_est,run_status,total_model_gflops_est,local_total_latency_ms
0,E607_1,E607,S3_C3_E607_V0568,test,0,44.375,46.375,profile_only,skipped,None,...,267.2334,0,0.002867,2,5.657237,False,0.00000,ok,5.657237,275.5222
1,E297_0,E297,S2_C2_E297_V0048,test,0,29.125,31.125,profile_only,executed,None,...,6.3713,1,0.700919,2,5.657237,True,1.86858,ok,7.525818,7938.8546
2,E549_1,E549,S3_C3_E549_V0444,test,0,29.250,31.250,profile_only,skipped,None,...,8.1576,0,0.004461,2,5.657237,False,0.00000,ok,5.657237,14.7431
3,E507_1,E507,S3_C3_E507_V0292,test,0,29.250,31.250,profile_only,skipped,None,...,13.8357,0,0.004042,2,5.657237,False,0.00000,ok,5.657237,19.7372
4,E504_0,E504,S3_C3_E504_V0288,test,0,14.125,16.125,profile_only,skipped,None,...,8.2659,0,0.097211,2,5.657237,False,0.00000,ok,5.657237,14.6929


In [13]:
print("Processing statuses:")
print(results.run_status.value_counts(dropna=False).to_string())
if "error" in results.columns:
    display(results.loc[results.run_status != "ok", ["sample_id", "error"]])

ok = results.loc[results.run_status == "ok"].copy()
assert len(ok), "Nothing completed. Inspect logged errors above."
print(f"Completed: {len(ok)}/{len(results)}")
print("Video mode:", VIDEO_INFERENCE_MODE, "— activity predictions meaningful?", VIDEO_INFERENCE_MODE == "trained")
print(f"Video activation rate: {ok.video_activated.mean():.1%}")
print(f"Executed video inference: {(ok.video_status == 'executed').sum()}")
print("Audio exits:", ok.exit_stage.value_counts().sort_index().to_dict())
print(f"Mean combined model GFLOPs (THOP ×2): {ok.total_model_gflops_est.mean():.3f}")
print(f"Video-only fixed-{NUM_FRAMES}-frame reference GFLOPs: {VIDEO_GFLOPS:.3f}")
print(f"Mean local latency incl. audio preprocessing + video decode: {ok.local_total_latency_ms.mean():.1f} ms")
if (ok.binary_label == 1).any():
    print(f"Animal recall of audio gate: {ok.loc[ok.binary_label == 1,'audio_prediction'].mean():.1%}")
if (ok.binary_label == 0).any():
    print(f"Background rejection of audio gate: {1-ok.loc[ok.binary_label == 0,'audio_prediction'].mean():.1%}")
if (ok.video_status == "executed").any():
    print(f"Average frames when vision executes: {ok.loc[ok.video_status == 'executed','video_frames_used'].mean():.2f}")
assert not ((ok.audio_prediction == 0) & (ok.video_status == "executed")).any()
assert not ((ok.audio_prediction == 1) & (ok.video_status != "executed")).any()
if VIDEO_INFERENCE_MODE == "profile_only":
    assert ok.activity_prediction.isna().all(), "Profiling-only must not output activity predictions"
    print("VIDEO OUTPUTS ARE FROM UNTRAINED WEIGHTS: no activity-quality claims")
if MAX_SAMPLES is not None:
    print("PILOT ONLY: increase MAX_SAMPLES for a representative workload")
print("No activity accuracy is claimed on unmatched raw frames")


Processing statuses:
run_status
ok    12
Completed: 12/12
Video mode: profile_only — activity predictions meaningful? False
Video activation rate: 58.3%
Executed video inference: 7
Audio exits: {1: 5, 2: 7}
Mean combined model GFLOPs (THOP ×2): 5.565
Video-only fixed-4-frame reference GFLOPs: 1.869
Mean local latency incl. audio preprocessing + video decode: 2389.6 ms
Animal recall of audio gate: 100.0%
Background rejection of audio gate: 83.3%
Average frames when vision executes: 4.00
VIDEO OUTPUTS ARE FROM UNTRAINED WEIGHTS: no activity-quality claims
PILOT ONLY: increase MAX_SAMPLES for a representative workload
No activity accuracy is claimed on unmatched raw frames


In [14]:
run_info = {
    "objective": "Stage A conditional model-execution measurement; activity accuracy not evaluated",
    "split": EVALUATION_SPLIT, "num_selected": len(selected),
    "num_completed": int((results.run_status == "ok").sum()),
    "audio_policy": "balanced", "audio_margin1": MARGIN1, "audio_margin2": MARGIN2,
    "audio_thresholds": {"exit1": T1, "exit2": T2, "final": T_FINAL},
    "video_architecture": "MobileNetV3 Large, frame-logit average",
    "video_inference_mode": VIDEO_INFERENCE_MODE,
    "video_checkpoint": str(VIDEO_CHECKPOINT) if VIDEO_WEIGHTS_AVAILABLE and VIDEO_INFERENCE_MODE == "trained" else None,
    "activity_predictions_valid": VIDEO_INFERENCE_MODE == "trained",
    "video_adaptive_threshold": VIDEO_ADAPTIVE_THRESHOLD,
    "video_num_frames": NUM_FRAMES,
    "frame_sampling": "nested positions inside 2-second window" if VIDEO_ADAPTIVE_THRESHOLD is not None else "interior evenly spaced within 2-second window",
    "flops": "THOP MACs*2, sum actual audio exit and video frames executed; excludes decoding/preprocessing",
    "audio_source": str(MANIFEST_PATH.relative_to(ROOT)),
    "torch": str(torch.__version__), "device": str(DEVICE),
    "download_missing_mp4": bool(DOWNLOAD_MISSING_MP4),
    "video_training_overlap_checked": bool(b1_train_path.exists()),
    "prediction_csv": str(OUTPUT_PATH.relative_to(ROOT)),
}
run_json = OUTPUT_PATH.with_suffix(".json")
run_json.write_text(json.dumps(run_info, indent=2), encoding="utf-8")
print("Saved run settings:", run_json)


Saved run settings: C:\Users\benja\Documents\CS_Master\Big Data at the Edge\Big_Data_at_the_Edge_Project\Audio\results\stage_a\stage_a_predictions_test_12_profile_only.json
